# Stack Overflow Developer Survey - Data Preprocessing

This notebook demonstrates the complete data preprocessing pipeline for Stack Overflow Developer Survey Analysis, including technology skill parsing, feature engineering, and dataset preparation for machine learning.

## 1. Import Required Libraries

Import pandas, numpy, sklearn, and other necessary libraries for data preprocessing and feature engineering.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load and Explore Dataset

Load the Stack Overflow survey dataset and perform initial exploration including shape, columns, data types, and missing values analysis.

In [3]:
# Load the dataset
file_path = "e:/ITProject/predSeeker/data/raw/stackoverflow_with_nulls.csv"
df = pd.read_csv(file_path)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset loaded successfully!
Shape: (73462, 15)
Columns: ['Unnamed: 0', 'Age', 'Accessibility', 'EdLevel', 'Employment', 'Gender', 'MentalHealth', 'MainBranch', 'YearsCode', 'YearsCodePro', 'Country', 'PreviousSalary', 'HaveWorkedWith', 'ComputerSkills', 'Employed']


In [3]:
# Explore data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nFirst few rows:")
df.head()

Data Types:
Unnamed: 0          int64
Age                object
Accessibility      object
EdLevel            object
Employment          int64
Gender             object
MentalHealth       object
MainBranch         object
YearsCode           int64
YearsCodePro        int64
Country            object
PreviousSalary    float64
HaveWorkedWith     object
ComputerSkills      int64
Employed            int64
dtype: object

Missing Values:
Unnamed: 0           0
Age                  0
Accessibility        0
EdLevel              0
Employment           0
Gender               0
MentalHealth         0
MainBranch           0
YearsCode            0
YearsCodePro         0
Country              0
PreviousSalary       0
HaveWorkedWith    3699
ComputerSkills       0
Employed             0
dtype: int64

First few rows:


,Unnamed: 0,Age,Accessibility,EdLevel,Employment,Gender,MentalHealth,MainBranch,YearsCode,YearsCodePro,Country,PreviousSalary,HaveWorkedWith,ComputerSkills,Employed
0,0,<35,No,Master,1,Man,No,Dev,7,4,Sweden,51552.0,C++;Python;Git;PostgreSQL,4,0
1,1,<35,No,Undergraduate,1,Man,No,Dev,12,5,Spain,46482.0,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,12,1
2,2,<35,No,Master,1,Man,No,Dev,15,6,Germany,77290.0,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,7,0
3,3,<35,No,Undergraduate,1,Man,No,Dev,9,6,Canada,46135.0,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,13,0
4,4,>35,No,PhD,0,Man,No,NotDev,40,30,Singapore,160932.0,C++;Python,2,0


## 3. Process HaveWorkedWith Column

Parse the semicolon-separated technology strings in HaveWorkedWith column and categorize technologies into skill families (Programming, Web, Database, CloudDevOps).

In [ ]:
# Define skill families for technology categorization
SKILL_FAMILIES = {
    'Programming': [
        'Python', 'Java', 'JavaScript', 'C++', 'C#', 'C', 'PHP', 'Ruby', 
        'Go', 'Rust', 'Swift', 'Kotlin', 'Scala', 'R', 'Matlab', 'Perl',
        'TypeScript', 'Dart', 'F#', 'Assembly', 'Delphi', 'VBA'
    ],
    'Web': [
        'HTML/CSS', 'React.js', 'Angular', 'Vue.js', 'Node.js', 'Express',
        'jQuery', 'Angular.js', 'Svelte', 'Django', 'Flask', 'Laravel',
        'Ruby on Rails', 'ASP.NET', 'ASP.NET Core', 'Spring', 'FastAPI'
    ],
    'Database': [
        'MySQL', 'PostgreSQL', 'MongoDB', 'SQLite', 'Redis', 'Oracle',
        'Microsoft SQL Server', 'MariaDB', 'DynamoDB', 'Elasticsearch',
        'Couchbase', 'Firebase', 'SQL'
    ],
    'CloudDevOps': [
        'AWS', 'Microsoft Azure', 'Google Cloud Platform', 'Docker', 
        'Kubernetes', 'Git', 'Terraform', 'Ansible', 'Heroku',
        'DigitalOcean', 'Bash/Shell', 'PowerShell'
    ]
}

print("Skill families defined:")
for family, techs in SKILL_FAMILIES.items():
    print(f"{family}: {len(techs)} technologies")

Skill families defined:
Programming: 22 technologies
Web: 17 technologies
Database: 13 technologies
CloudDevOps: 12 technologies


In [5]:
def parse_technologies(tech_string):
    """Parse the HaveWorkedWith column into a list of technologies"""
    if pd.isna(tech_string) or tech_string == '' or tech_string == 'Unknown':
        return []
    
    # Split by semicolon and clean up
    techs = [tech.strip() for tech in str(tech_string).split(';') if tech.strip()]
    return techs

# Parse technologies for each row
df['Technologies_List'] = df['HaveWorkedWith'].apply(parse_technologies)

print("Technology parsing completed!")
print(f"Sample parsed technologies: {df['Technologies_List'].iloc[0]}")

Technology parsing completed!
Sample parsed technologies: ['C++', 'Python', 'Git', 'PostgreSQL']


## 4. Handle Missing Values

Implement strategies to handle missing values: fill categorical columns with mode, numerical columns with median, and special handling for ComputerSkills column.

In [ ]:
print("Missing values before cleaning:")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

# Handle missing values by column type
categorical_cols = ['Age', 'Accessibility', 'EdLevel', 'Gender', 'MentalHealth', 'MainBranch', 'Country']
for col in categorical_cols:
    if col in df.columns:
        mode_value = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
        df[col] = df[col].fillna(mode_value)
        print(f"[OK] {col}: Filled with '{mode_value}'")

# Numerical columns - fill with median
numerical_cols = ['YearsCode', 'YearsCodePro', 'PreviousSalary']
for col in numerical_cols:
    if col in df.columns:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)
        print(f"[OK] {col}: Filled with {median_value}")
        
# ComputerSkills - special handling
if 'ComputerSkills' in df.columns:
    df['Parsed_Tech_Count'] = df['Technologies_List'].apply(len)
    df['ComputerSkills'] = df['ComputerSkills'].fillna(df['Parsed_Tech_Count'])
    print("[OK] ComputerSkills: Filled with parsed technology count where missing")

Missing values before cleaning:
HaveWorkedWith    3699
dtype: int64
[OK] Age: Filled with '<35'
[OK] Accessibility: Filled with 'No'
[OK] EdLevel: Filled with 'Undergraduate'
[OK] Gender: Filled with 'Man'
[OK] MentalHealth: Filled with 'No'
[OK] MainBranch: Filled with 'Dev'
[OK] Country: Filled with 'United States of America'
[OK] YearsCode: Filled with 12.0
[OK] YearsCodePro: Filled with 7.0
[OK] PreviousSalary: Filled with 57588.0
[OK] ComputerSkills: Filled with parsed technology count where missing


## 5. Create Technology Features

Calculate skill family scores (percentage of technologies known in each family), binary flags for each skill family, skill breadth, and full-stack developer indicator.

In [8]:
def calculate_skill_family_scores(tech_list):
    """Calculate percentage scores for each skill family"""
    scores = {}
    binary_flags = {}
    
    for family_name, family_techs in SKILL_FAMILIES.items():
        # Count technologies from this family
        known_techs = [tech for tech in tech_list if tech in family_techs]
        known_count = len(known_techs)
        total_count = len(family_techs)
        
        # Calculate percentage score
        percentage = (known_count / total_count) * 100 if total_count > 0 else 0
        
        # Store results
        scores[f'{family_name}_Score'] = round(percentage, 1)
        binary_flags[f'Has_{family_name}'] = 1 if known_count > 0 else 0
    
    return scores, binary_flags

def calculate_derived_features(binary_flags):
    """Calculate skill breadth and full-stack indicator"""
    skill_breadth = sum(binary_flags.values())
    
    is_fullstack = 1 if (binary_flags['Has_Programming'] == 1 and 
                        binary_flags['Has_Web'] == 1 and 
                        binary_flags['Has_Database'] == 1) else 0
    
    return {
        'Skill_Breadth': skill_breadth,
        'Is_FullStack': is_fullstack
    }

# Process each row to create technology features
all_scores = []
all_binary_flags = []
all_derived = []

for tech_list in df['Technologies_List']:
    scores, binary_flags = calculate_skill_family_scores(tech_list)
    derived = calculate_derived_features(binary_flags)
    
    all_scores.append(scores)
    all_binary_flags.append(binary_flags)
    all_derived.append(derived)

# Convert to DataFrames and merge
scores_df = pd.DataFrame(all_scores)
binary_df = pd.DataFrame(all_binary_flags)
derived_df = pd.DataFrame(all_derived)

# Add new columns to main dataframe
df = pd.concat([df, scores_df, binary_df, derived_df], axis=1)

print("Technology features created!")
print(f"Added {len(scores_df.columns) + len(binary_df.columns) + len(derived_df.columns)} new columns")
print("\nNew technology features:")
tech_features = list(scores_df.columns) + list(binary_df.columns) + list(derived_df.columns)
print(tech_features)

Technology features created!
Added 10 new columns

New technology features:
['Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score', 'Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps', 'Skill_Breadth', 'Is_FullStack']


## 6. Create Additional Features

Engineer additional features including age groups, education level encoding, developer status, mental health indicators, accessibility needs, gender encoding, and experience features.

In [7]:
# Age group (binary)
df['IsYoung'] = (df['Age'] == '<35').astype(int)
print(f"[OK] IsYoung: {df['IsYoung'].sum():,} young developers")

# Education level (ordinal)
education_order = {
    'NoHigherEd': 0, 'Other': 1, 'Undergraduate': 2, 'Master': 3, 'PhD': 4
}
df['EducationLevel_Numeric'] = df['EdLevel'].map(education_order).fillna(1)
print("[OK] EducationLevel_Numeric: Ordinal education encoding")

# Developer status
df['IsDeveloper'] = (df['MainBranch'] == 'Dev').astype(int)
print(f"[OK] IsDeveloper: {df['IsDeveloper'].sum():,} developers")

# Mental health
df['HasMentalHealthConcerns'] = (df['MentalHealth'] == 'Yes').astype(int)
print(f"[OK] HasMentalHealthConcerns: {df['HasMentalHealthConcerns'].sum():,} with concerns")

# Accessibility needs
df['HasAccessibilityNeeds'] = (df['Accessibility'] == 'Yes').astype(int)
print(f"[OK] HasAccessibilityNeeds: {df['HasAccessibilityNeeds'].sum():,} with accessibility needs")

[OK] IsYoung: 47,819 young developers
[OK] EducationLevel_Numeric: Ordinal education encoding
[OK] IsDeveloper: 67,396 developers
[OK] HasMentalHealthConcerns: 16,518 with concerns
[OK] HasAccessibilityNeeds: 2,107 with accessibility needs


In [ ]:
# Gender (one-hot encoding)
print("Gender distribution:")
print(df['Gender'].value_counts())

df['Gender_Man'] = (df['Gender'] == 'Man').astype(int)
df['Gender_Woman'] = (df['Gender'] == 'Woman').astype(int)
df['Gender_NonBinary'] = (df['Gender'] == 'NonBinary').astype(int)
print(f"\nGender features created:")
print(f"- Gender_Man: {df['Gender_Man'].sum():,}")
print(f"- Gender_Woman: {df['Gender_Woman'].sum():,}")
print(f"- Gender_NonBinary: {df['Gender_NonBinary'].sum():,}")

# Experience features
df['HasProfessionalExperience'] = (df['YearsCodePro'] > 0).astype(int)
print(f"[OK] HasProfessionalExperience: {df['HasProfessionalExperience'].sum():,}")

# Salary features
df['HasSalaryInfo'] = (~df['PreviousSalary'].isna()).astype(int)
print(f"[OK] HasSalaryInfo: {df['HasSalaryInfo'].sum():,} with salary info")

Gender distribution:
Gender
Man          68573
Woman         3518
NonBinary     1371
Name: count, dtype: int64

Gender features created:
- Gender_Man: 68,573
- Gender_Woman: 3,518
- Gender_NonBinary: 1,371
[OK] HasProfessionalExperience: 70,519
[OK] HasSalaryInfo: 73,462 with salary info


## 7. Drop Unwanted Features

 Remove features with low predictive capacity, prevent data leakage, and eliminate redundant columns to optimize the feature set for machine learning.


In [ ]:
print("🗑️ FEATURE DROPPING - INDIVIDUAL ANALYSIS")
print("=" * 60)

# Show initial dataset state
print(f"📊 Initial dataset shape: {df.shape}")
print(f"📋 Initial columns: {len(df.columns)}")

# 1️⃣ DROP INDEX COLUMN
print("\n" + "="*50)
print("1️⃣ DROPPING INDEX COLUMN")
print("="*50)

if 'Unnamed: 0' in df.columns:
    print("   📌 Column: 'Unnamed: 0'")
    # Drop the column
    df = df.drop(columns=['Unnamed: 0'])
    print(f"   ✅ DROPPED! New shape: {df.shape}")
else:
    print("   ℹ️  'Unnamed: 0' column not found in dataset")

# 2 DROP TOTAL YEARS OF CODING
print("\n" + "="*50)
print("3️⃣ DROPPING TOTAL YEARS OF CODING")
print("="*50)

if 'YearsCode' in df.columns:
    print("   📌 Column: 'YearsCode'")
    print("   🎯 Reason: Extremely low correlation with employment")
    print("   📊 Correlation: 0.0020 (nearly zero)")
    
    # Show some statistics before dropping
    print(f"   📈 Statistics: Min={df['YearsCode'].min()}, Max={df['YearsCode'].max()}, Mean={df['YearsCode'].mean():.1f}")
    
    # Drop the column
    df = df.drop(columns=['YearsCode'])
    print(f"   ✅ DROPPED! New shape: {df.shape}")
else:
    print("   ℹ️  'YearsCode' column not found in dataset")

# 4️⃣ DROP COUNTRY
print("\n" + "="*50)
print("4️⃣ DROPPING COUNTRY")
print("="*50)

if 'Country' in df.columns:
    print("   📌 Column: 'Country'")
    print("   🎯 Reason: High cardinality causing dimensionality issues")
    
    # Drop the column
    df = df.drop(columns=['Country'])
    print(f"   ✅ DROPPED! New shape: {df.shape}")
else:
    print("   ℹ️  'Country' column not found in dataset")


🗑️ FEATURE DROPPING - INDIVIDUAL ANALYSIS
📊 Initial dataset shape: (73462, 11)
📋 Initial columns: 11

1️⃣ DROPPING INDEX COLUMN
   ℹ️  'Unnamed: 0' column not found in dataset

2️⃣ DROPPING YEARS OF PROFESSIONAL CODING
   ℹ️  'YearsCodePro' column not found in dataset

3️⃣ DROPPING TOTAL YEARS OF CODING
   ℹ️  'YearsCode' column not found in dataset

4️⃣ DROPPING COUNTRY
   ℹ️  'Country' column not found in dataset


## 7. Feature Selection and Optimization

Select final features for machine learning, drop low-predictive features, handle redundant columns, and optimize the feature set for employment prediction.

In [10]:
# Define final features for machine learning
tech_features = [
    'Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score',
    'Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps',
    'Skill_Breadth', 'Is_FullStack'
]

additional_features = [
    'ComputerSkills',  # Strong predictor
    'EducationLevel_Numeric', 'IsYoung', 'IsDeveloper',
    'HasMentalHealthConcerns', 'HasAccessibilityNeeds',
    'Gender_Man', 'Gender_Woman', 'Gender_NonBinary',
    'HasProfessionalExperience', 'HasSalaryInfo'
]

# Combine all features
selected_features = tech_features + additional_features
target_column = 'Employed'

print(f"Selected {len(selected_features)} features for machine learning:")
print("\nTechnology Features:")
for feature in tech_features:
    print(f"   - {feature}")
print("\nAdditional Features:")
for feature in additional_features:
    print(f"   - {feature}")

# Create final dataset
X = df[selected_features].copy()
y = df[target_column].copy()

print(f"\nFinal dataset shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts(normalize=True) * 100)

Selected 21 features for machine learning:

Technology Features:
   - Programming_Score
   - Web_Score
   - Database_Score
   - CloudDevOps_Score
   - Has_Programming
   - Has_Web
   - Has_Database
   - Has_CloudDevOps
   - Skill_Breadth
   - Is_FullStack

Additional Features:
   - ComputerSkills
   - EducationLevel_Numeric
   - IsYoung
   - IsDeveloper
   - HasMentalHealthConcerns
   - HasAccessibilityNeeds
   - Gender_Man
   - Gender_Woman
   - Gender_NonBinary
   - HasProfessionalExperience
   - HasSalaryInfo

Final dataset shape: (73462, 21)
Target distribution:
Employed
1    53.622281
0    46.377719
Name: proportion, dtype: float64


## 8. Data Splitting and Scaling

Split data into training and test sets, apply StandardScaler to numerical features, and prepare the final dataset for machine learning models.

In [11]:
# Handle any remaining missing or infinite values
X = X.fillna(X.median())
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print("Final data cleaning completed!")
print(f"Missing values in X: {X.isnull().sum().sum()}")
print(f"Missing values in y: {y.isnull().sum()}")

Final data cleaning completed!
Missing values in X: 0
Missing values in y: 0


In [12]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
numerical_features = X.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print("Data preparation completed!")
print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Features: {X.shape[1]}")

print(f"\nTarget distribution in training set:")
print((y_train.value_counts(normalize=True) * 100).round(2))

Data preparation completed!
Training set: (58769, 21)
Test set: (14693, 21)
Features: 21

Target distribution in training set:
Employed
1    53.62
0    46.38
Name: proportion, dtype: float64


In [14]:
# Display final feature summary
print("PREPROCESSING COMPLETED!")
print("=" * 50)
print(f"Original dataset: {df.shape}")
print(f"ML feature set: {X.shape}")
print(f"Features selected: {X.shape[1]}")
print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")

# Show feature correlation with target (if needed for demonstration)
feature_correlations = X.corrwith(y).sort_values(ascending=False)
print(f"\nTop 10 features by correlation with employment:")
print(feature_correlations.head(10).round(4))

PREPROCESSING COMPLETED!
Original dataset: (73462, 37)
ML feature set: (73462, 21)
Features selected: 21
Training samples: 58769
Test samples: 14693

Top 10 features by correlation with employment:
ComputerSkills       0.5855
Web_Score            0.5323
Database_Score       0.4308
Programming_Score    0.4002
Is_FullStack         0.3968
Has_Web              0.3530
Skill_Breadth        0.2710
CloudDevOps_Score    0.2461
Has_Database         0.2350
Has_CloudDevOps      0.1240
dtype: float64


In [15]:
# Save processed data for future use
X_train_scaled.to_csv("e:/ITProject/predSeeker/data/processed/X_train_scaled.csv", index=False)
X_test_scaled.to_csv("e:/ITProject/predSeeker/data/processed/X_test_scaled.csv", index=False)
y_train.to_csv("e:/ITProject/predSeeker/data/processed/y_train.csv", index=False)
y_test.to_csv("e:/ITProject/predSeeker/data/processed/y_test.csv", index=False)

print("Processed datasets saved successfully!")
print("Files saved:")
print("- X_train_scaled.csv")
print("- X_test_scaled.csv") 
print("- y_train.csv")
print("- y_test.csv")

Processed datasets saved successfully!
Files saved:
- X_train_scaled.csv
- X_test_scaled.csv
- y_train.csv
- y_test.csv
